#**ETL DATASET AVES**


---





#Carga de archivo

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import re, os, unicodedata

# Montar Google Drive para acceder al archivo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Ruta del archivo en Google Drive
# Asegúrate de que la ruta sea correcta.
DATA_DIR = '/content/drive/MyDrive/Proyectos/Aves'

PATH_INPUT = f'{DATA_DIR}/dataset-2025.csv'  # must contain latitude, longitude
PATH_OUTPUT = f'{DATA_DIR}/dataset-2025-definitivo.csv'

# Cargar el archivo CSV en un DataFrame de pandas
try:
    df = pd.read_csv(PATH_INPUT, sep='\t', engine='python')
    df_copy = df.copy(deep=True)
    print("Archivo cargado exitosamente.")
    print(df.head()) # Mostrar las primeras filas del DataFrame
    print(df.describe())
    print(df.info())
except FileNotFoundError:
    print(f"Error: El archivo no se encontró en la ruta especificada: {PATH_INPUT}")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo: {e}")

Mounted at /content/drive
Archivo cargado exitosamente.
       gbifID                            datasetKey  \
0  5858379769  e3ce628e-9683-4af7-b7a9-47eef785d3bb   
1  5858386191  e3ce628e-9683-4af7-b7a9-47eef785d3bb   
2  5063633106  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
3  5230239878  50c9509d-22c7-4a22-a47d-8c48425ef4a7   
4  5838274528  50c9509d-22c7-4a22-a47d-8c48425ef4a7   

                                        occurrenceID   kingdom    phylum  \
0               24d7b1a0-7301-4c0f-9e84-de21eb545ef6  Animalia  Chordata   
1               f7d4c062-2e59-4029-ba6a-b3004e920df5  Animalia  Chordata   
2  https://www.inaturalist.org/observations/26024...  Animalia  Chordata   
3  https://www.inaturalist.org/observations/29764...  Animalia  Chordata   
4  https://www.inaturalist.org/observations/31784...  Animalia  Chordata   

  class              order        family         genus  \
0  Aves  Procellariiformes   Diomedeidae      Diomedea   
1  Aves  Procellariiformes   Diomedeidae

# Eliminación de valores nulo y columnas no útiles

In [50]:
# Eliminar columnas con todos los valores nulos
df.dropna(axis=1, how='all', inplace=True)

# Eliminar columnas no útiles
df.drop(columns=['gbifID', 'datasetKey', 'occurrenceID', 'infraspecificEpithet',
                 'scientificName', 'countryCode', 'occurrenceStatus',
                 'publishingOrgKey', 'taxonKey', 'speciesKey',
                 'catalogNumber', 'identifiedBy', 'dateIdentified',
                 'license', 'rightsHolder', 'recordedBy', 'lastInterpreted',
                 'issue', 'recordNumber', 'verbatimScientificNameAuthorship'], inplace=True)

# Verificar las columnas restantes y sus valores no nulos
print("DataFrame después de eliminar columnas con todos los valores nulos:")
print(df.info())

DataFrame después de eliminar columnas con todos los valores nulos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85131 entries, 0 to 85130
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   kingdom                        85131 non-null  object 
 1   phylum                         85131 non-null  object 
 2   class                          85131 non-null  object 
 3   order                          85131 non-null  object 
 4   family                         85131 non-null  object 
 5   genus                          85114 non-null  object 
 6   species                        83491 non-null  object 
 7   taxonRank                      85131 non-null  object 
 8   verbatimScientificName         85131 non-null  object 
 9   locality                       1917 non-null   object 
 10  stateProvince                  84608 non-null  object 
 11  individualCount                1227 no

In [51]:
for columna in df.columns:
    conteo = df[columna].value_counts()
    print(f"\nConteo de valores en la columna '{columna}':")
    print(conteo)


Conteo de valores en la columna 'kingdom':
kingdom
Animalia    85131
Name: count, dtype: int64

Conteo de valores en la columna 'phylum':
phylum
Chordata    85131
Name: count, dtype: int64

Conteo de valores en la columna 'class':
class
Aves    85131
Name: count, dtype: int64

Conteo de valores en la columna 'order':
order
Passeriformes          38815
Anseriformes            7119
Charadriiformes         5420
Pelecaniformes          4245
Falconiformes           4024
Columbiformes           4013
Accipitriformes         3752
Piciformes              2505
Gruiformes              2215
Psittaciformes          2092
Apodiformes             1697
Strigiformes            1577
Suliformes              1516
Podicipediformes        1086
Cuculiformes             940
Procellariiformes        692
Ciconiiformes            651
Phoenicopteriformes      625
Coraciiformes            573
Sphenisciformes          318
Galliformes              287
Rheiformes               285
Tinamiformes             277
Caprimu

In [52]:
# Eliminar columnas de reino, filo y clase (son todas iguales ya que son aves)
df.drop(columns=['kingdom', 'phylum', 'class'], inplace=True)

# Eliminar columna locality (>90% de nulos)
df.drop(columns=['locality'], inplace=True)

# Eliminar otras columnas no útiles
df.drop(columns=['basisOfRecord', 'institutionCode', 'collectionCode',
                 'mediaType'], inplace=True)

# Verificar las columnas restantes y sus valores no nulos
print("DataFrame después de eliminar columnas con todos los valores nulos:")
print(df.info())

DataFrame después de eliminar columnas con todos los valores nulos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85131 entries, 0 to 85130
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order                          85131 non-null  object 
 1   family                         85131 non-null  object 
 2   genus                          85114 non-null  object 
 3   species                        83491 non-null  object 
 4   taxonRank                      85131 non-null  object 
 5   verbatimScientificName         85131 non-null  object 
 6   stateProvince                  84608 non-null  object 
 7   individualCount                1227 non-null   float64
 8   decimalLatitude                85117 non-null  float64
 9   decimalLongitude               85117 non-null  float64
 10  coordinateUncertaintyInMeters  67714 non-null  float64
 11  eventDate                      85131 n

In [53]:
# Eliminar registros con clasificaciones atípicas en taxonRank
df = df.loc[df["taxonRank"] != 'FAMILY']

# Asignar el valor de scientificName a registros sin valor en species
df.loc[df['species'].isnull(), 'species'] = df['verbatimScientificName']

# Eliminar columna taxonRank
df.drop(columns=['taxonRank', 'verbatimScientificName'], inplace=True)

# Verificar las columnas restantes y sus valores no nulos
print("DataFrame después de tratamiento de taxonRank:")
print(df.info())
for columna in df.columns:
    conteo = df[columna].value_counts()
    print(f"\nConteo de valores en la columna '{columna}':")
    print(conteo)

DataFrame después de tratamiento de taxonRank:
<class 'pandas.core.frame.DataFrame'>
Index: 85114 entries, 0 to 85130
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order                          85114 non-null  object 
 1   family                         85114 non-null  object 
 2   genus                          85114 non-null  object 
 3   species                        85114 non-null  object 
 4   stateProvince                  84591 non-null  object 
 5   individualCount                1227 non-null   float64
 6   decimalLatitude                85100 non-null  float64
 7   decimalLongitude               85100 non-null  float64
 8   coordinateUncertaintyInMeters  67703 non-null  float64
 9   eventDate                      85114 non-null  object 
 10  day                            85114 non-null  int64  
 11  month                          85114 non-null  int64  
 12  year

In [54]:
df.to_csv('dataset-filtrado.csv', index=False)

# Imputación de valores para reemplazar nulos

In [55]:
# Media
media = df["coordinateUncertaintyInMeters"].mean()

# Deciles (quantile divide en percentiles, 0.1=10%, 0.2=20%, etc.)
deciles = df["coordinateUncertaintyInMeters"].quantile([i/10 for i in range(1,10)])

print("Media:", media)
print("\nDeciles:")
print(deciles)

Media: 4494.493345937403

Deciles:
0.1       4.0
0.2      13.0
0.3      31.0
0.4      61.0
0.5     139.0
0.6     240.0
0.7     555.0
0.8    2503.0
0.9    6716.6
Name: coordinateUncertaintyInMeters, dtype: float64


In [56]:
# Asignar valor 1 en individualCount a todos los registros sin especificarlo
df["individualCount"] = df["individualCount"].fillna(1)

# Asignar la mediana en uncertainity para valores nulos
df["coordinateUncertaintyInMeters"] = df["coordinateUncertaintyInMeters"].fillna(df["coordinateUncertaintyInMeters"].median())

# Tratamiento de fechas y horas

In [57]:
# Fragmento: extrae hora desde df["eventDate"] y crea df["time"]
# Asume que `df` y `pd` ya están definidos en el notebook.
# Cambia TIME_FORMAT o CONVERT_TO_TZ según necesites.

import re

TIME_FORMAT = "HH:MM"        # opciones: "HH:MM:SS" o "HH:MM"
CONVERT_TO_TZ = "America/Argentina/Buenos_Aires"            # e.g. "America/Argentina/Buenos_Aires" o None

_presence_re = re.compile(r'[T\s]\d{2}:\d{2}')
_extract_re = re.compile(r'[T\s](\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?)')

def extract_time_from_eventdate(val, time_format=TIME_FORMAT, convert_to_tz=CONVERT_TO_TZ):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip()
    # si no hay indicio de tiempo (ni 'T' ni espacio seguido de HH:MM) -> NA
    if not _presence_re.search(s):
        return pd.NA

    # intento principal: pandas (maneja Z y offsets)
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    if pd.isna(dt):
        # fallback por regex (captura tras 'T' o espacio)
        m = _extract_re.search(s)
        if not m:
            return pd.NA
        time_str = m.group(1).split('.')[0]  # quitar fracciones de segundo si existen
        # normalizar según formato pedido
        if time_format == "HH:MM":
            return time_str[:5]
        # si faltan segundos, agregar ":00"
        return time_str if time_str.count(":") == 2 else time_str + ":00"

    # si se pidió conversión de zona horaria la aplicamos
    if convert_to_tz:
        try:
            dt = dt.tz_convert(convert_to_tz)
        except Exception:
            return pd.NA

    t = dt.time().replace(microsecond=0)
    return t.strftime("%H:%M") if time_format == "HH:MM" else t.strftime("%H:%M:%S")

def time_to_range(t):
    """
    Convierte un valor de df['time'] (p. ej. "19:18:15" o datetime.time) a:
      1 -> 00:00-05:59
      2 -> 06:00-11:59
      3 -> 12:00-17:59
      4 -> 18:00-23:59
    Devuelve pd.NA si t es NaN, vacío o no parseable.
    """
    if pd.isna(t):
        return pd.NA
    try:
        # Si es un objeto time/Timestamp con atributo hour
        if hasattr(t, "hour"):
            hour = int(t.hour)
        else:
            s = str(t).strip()
            if not s:
                return pd.NA
            hour = int(s.split(":")[0])
    except Exception:
        return pd.NA

    if 0 <= hour < 6:
        return 1
    if 6 <= hour < 12:
        return 2
    if 12 <= hour < 18:
        return 3
    if 18 <= hour < 24:
        return 4
    return pd.NA

# Aplicar al DataFrame (vectorizado con apply)
df["time"] = df["eventDate"].apply(lambda v: extract_time_from_eventdate(v, TIME_FORMAT, CONVERT_TO_TZ))

# Eliminar columnas con time nulo
df.dropna(subset=["time"], inplace=True)

# Aplicación al DataFrame (usa dtype nullable Int64 para mantener NA)
df["timeRange"] = df["time"].apply(time_to_range).astype("Int64")

# Display the first few rows with the new 'time' column
display(df[['eventDate', 'time', 'timeRange']].head())

# Check for null values in the new 'time' column
print("\nNull values in 'time' column:")
print(df['time'].isnull().sum())

# Eliminar la columna eventDate
df.drop(columns=['eventDate'], inplace=True)

,eventDate,time,timeRange
0,2025-04-03T23:35Z,20:35,4
1,2025-04-03T23:33Z,20:33,4
2,2025-01-31T19:39,16:39,3
3,2025-07-12T17:20:42,14:20,3
4,2025-09-15T19:10,16:10,3



Null values in 'time' column:
0


In [58]:
import numpy as np

# Convertir la columna 'time' a formato de hora decimal
df['hour'] = df['time'].str.split(':').apply(lambda x: int(x[0]) + int(x[1])/60)

# Aplicar transformación cíclica para la hora del día
df['hour_cycle'] = np.cos(2 * np.pi * df['hour'] / 24)

# Mostrar las primeras filas con las nuevas columnas
display(df[['time', 'hour', 'hour_cycle']].head())

# Verificar los tipos de datos y valores no nulos
print("\nDataFrame después de agregar 'hour' y 'hour_cycle':")
print(df.info())

,time,hour,hour_cycle
0,20:35,20.583333,0.625923
1,20:33,20.550000,0.619094
2,16:39,16.650000,-0.346117
3,14:20,14.333333,-0.819152
4,16:10,16.166667,-0.461749



DataFrame después de agregar 'hour' y 'hour_cycle':
<class 'pandas.core.frame.DataFrame'>
Index: 82446 entries, 0 to 83898
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order                          82446 non-null  object 
 1   family                         82446 non-null  object 
 2   genus                          82446 non-null  object 
 3   species                        82446 non-null  object 
 4   stateProvince                  81990 non-null  object 
 5   individualCount                82446 non-null  float64
 6   decimalLatitude                82446 non-null  float64
 7   decimalLongitude               82446 non-null  float64
 8   coordinateUncertaintyInMeters  82446 non-null  float64
 9   day                            82446 non-null  int64  
 10  month                          82446 non-null  int64  
 11  year                           82446 non-null  int64  
 12

# Transformación de Provincias
Primero vamos a ver la cantidad de avistamientos que tenemos para cada provincia

In [59]:
conteo = df["stateProvince"].value_counts()

print(conteo)

stateProvince
Buenos Aires                       18777
La Pampa                           13142
Santa Fe                            9503
Ciudad de Buenos Aires              8858
Entre Ríos                          5496
Misiones                            4149
Córdoba                             3557
Tierra del Fuego                    2377
Santa Cruz                          1681
Jujuy                               1568
Río Negro                           1521
Corrientes                          1376
Neuquén                             1269
Formosa                             1242
Catamarca                           1235
Chubut                              1189
Mendoza                              934
Tucumán                              908
San Luis                             843
Salta                                803
Autonomous City of Buenos Aires      453
Chaco                                347
Santiago del Estero                  275
San Juan                             237
La

Ahora eliminamos las filas que contengan provincias que no correspondan a Argentina

In [60]:
# Eliminar columnas con stateProvince nulo
df.dropna(subset=["stateProvince"], inplace=True)

provincias_a_eliminar = ["Artigas", "Magallanes y Antártica Chilena", "Asunción"]

# Filtro para quitar esas filas
df = df[~df["stateProvince"].isin(provincias_a_eliminar)]

print(df["stateProvince"].value_counts())

stateProvince
Buenos Aires                       18777
La Pampa                           13142
Santa Fe                            9503
Ciudad de Buenos Aires              8858
Entre Ríos                          5496
Misiones                            4149
Córdoba                             3557
Tierra del Fuego                    2377
Santa Cruz                          1681
Jujuy                               1568
Río Negro                           1521
Corrientes                          1376
Neuquén                             1269
Formosa                             1242
Catamarca                           1235
Chubut                              1189
Mendoza                              934
Tucumán                              908
San Luis                             843
Salta                                803
Autonomous City of Buenos Aires      453
Chaco                                347
Santiago del Estero                  275
San Juan                             237
La

Ahora reemplazamos algunos lugares por la provincia correspondiente

In [61]:
reemplazos = {
    "Buenos Aires (Province)": "Buenos Aires",
    "Santa Cruz Province, Argentina": "Santa Cruz",
    "Rio Negro":"Río Negro",
    "Alto Paraná": "Misiones",
    "Paraná": "Misiones",
    "Tierra del Fuego Province": "Tierra del Fuego",
    "Autonomous City of Buenos Aires": "Ciudad de Buenos Aires"
}

df.loc[:, "stateProvince"] = df["stateProvince"].replace(reemplazos)
print(df["stateProvince"].value_counts())

stateProvince
Buenos Aires              18777
La Pampa                  13142
Santa Fe                   9503
Ciudad de Buenos Aires     9311
Entre Ríos                 5496
Misiones                   4174
Córdoba                    3557
Tierra del Fuego           2384
Santa Cruz                 1712
Jujuy                      1568
Río Negro                  1521
Corrientes                 1376
Neuquén                    1269
Formosa                    1242
Catamarca                  1235
Chubut                     1189
Mendoza                     934
Tucumán                     908
San Luis                    843
Salta                       803
Chaco                       347
Santiago del Estero         275
San Juan                    237
La Rioja                    170
Name: count, dtype: int64


Luego vamos a asignarle un numero a cada provincia y creamos la columna llamada "provincia_num"

In [62]:
df["provincia_num"] = pd.factorize(df["stateProvince"])[0]

print(df.loc[:, ["stateProvince", "provincia_num"]].head(10))

   stateProvince  provincia_num
2       La Pampa              0
3   Buenos Aires              1
4          Salta              2
5       Santa Fe              3
6       Santa Fe              3
7       La Pampa              0
8     Entre Ríos              4
9       Misiones              5
10  Buenos Aires              1
11       Tucumán              6


# Tratamiento de year y day

Convertimos day, month, year e individualCount a int64, para respetar su naturaleza

In [63]:
# Convertir columnas a tipo int64
df['day'] = df['day'].astype('int64')
df['month'] = df['month'].astype('int64')
df['timeRange'] = df['timeRange'].astype('int64')
df['individualCount'] = df['individualCount'].astype('int64')

# Eliminar columnas year, time, stateProvince
df.drop(columns=['year', 'time', 'stateProvince'], inplace=True)

Agregamos una columna "day_of_year", que puede aportar al entrenamiento del modelo por la naturaleza cíclica de los días del año.

In [64]:
# Diccionario para mapear el mes al número acumulado de días antes de ese mes (en un año no bisiesto)
days_before_month = {
    1: 0,  # January
    2: 31, # February
    3: 59, # March
    4: 90, # April
    5: 120, # May
    6: 151, # June
    7: 181, # July
    8: 212, # August
    9: 243, # September
    10: 273, # October
    11: 304, # November
    12: 334  # December
}

# Calcular day_of_year
# Convert month and day to integers for dictionary lookup and calculation
df['day_of_year'] = df.apply(
    lambda row: days_before_month.get(int(row['month']), 0) + int(row['day']),
    axis=1
)

# Handle February 29th as the 60th day (as requested)
df.loc[(df['month'] == 2) & (df['day'] == 29), 'day_of_year'] = 60

# Verificar los cambios
print("DataFrame después de agregar la columna 'day_of_year':")
print(df[['day', 'month', 'day_of_year']].head())
print(df.info())

DataFrame después de agregar la columna 'day_of_year':
   day  month  day_of_year
2   31      1           31
3   12      7          193
4   15      9          258
5    6      7          187
6   13      7          194
<class 'pandas.core.frame.DataFrame'>
Index: 81973 entries, 2 to 83898
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order                          81973 non-null  object 
 1   family                         81973 non-null  object 
 2   genus                          81973 non-null  object 
 3   species                        81973 non-null  object 
 4   individualCount                81973 non-null  int64  
 5   decimalLatitude                81973 non-null  float64
 6   decimalLongitude               81973 non-null  float64
 7   coordinateUncertaintyInMeters  81973 non-null  float64
 8   day                            81973 non-null  int64  
 9   month         

# Integración de dataset externo

In [65]:
!pip install geopandas rasterio

In [66]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.warp import transform
from rasterio.mask import mask
import numpy as np
import os
from tqdm import tqdm

In [67]:
# Rasters (download once, then reuse)
# Use global rasters in GeoTIFF form (.tif)
PATH_ELEV = f'{DATA_DIR}/elevation_argentina.tif'
PATH_TEMP = f'{DATA_DIR}/mean_temp_argentina.tif'    # BIO1 = mean annual temperature
PATH_PREC = f'{DATA_DIR}/annual_prec_argentina.tif' # BIO12 = annual precipitation

In [68]:
# --- 1) Load coordinates ---

# Check your coordinate columns — adjust names if needed
lat_col, lon_col = 'decimalLatitude', 'decimalLongitude'

# Drop rows without valid coordinates
df = df.dropna(subset=[lat_col, lon_col])

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
    crs="EPSG:4326"
)
gdf.head()

,order,family,genus,species,individualCount,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,day,month,timeRange,hour,hour_cycle,provincia_num,day_of_year,geometry
2,Passeriformes,Fringillidae,Spinus,Spinus magellanicus,1,-35.659324,-63.757789,4571.0,31,1,3,16.650000,-0.346117,0,31,POINT (-63.75779 -35.65932)
3,Apodiformes,Trochilidae,Leucochloris,Leucochloris albicollis,1,-34.491839,-58.479915,20.0,12,7,3,14.333333,-0.819152,1,193,POINT (-58.47992 -34.49184)
4,Passeriformes,Parulidae,Setophaga,Setophaga pitiayumi,1,-25.276728,-65.387564,2.0,15,9,3,16.166667,-0.461749,2,258,POINT (-65.38756 -25.27673)
5,Anseriformes,Anatidae,Anas,Anas flavirostris,1,-31.554500,-61.052494,139.0,6,7,2,8.483333,-0.605294,3,187,POINT (-61.05249 -31.5545)
6,Piciformes,Picidae,Melanerpes,Melanerpes cactorum,1,-30.197865,-61.165918,4.0,13,7,2,8.800000,-0.669131,3,194,POINT (-61.16592 -30.19786)


In [69]:
# --- 2) Function to sample raster values at points ---
def extract_raster_values(gdf, raster_path, col_name):
    with rasterio.open(raster_path) as src:
        # Ensure same CRS
        gdf = gdf.to_crs(src.crs)
        # Extract pixel value for each point
        values = []
        for geom in tqdm(gdf.geometry, desc=f"Extracting {col_name}"):
            try:
                for val in src.sample([(geom.x, geom.y)]):
                    values.append(val[0])
            except Exception:
                values.append(np.nan)
        gdf[col_name] = values
    return gdf

In [70]:
# --- 3) Add Elevation ---
gdf = extract_raster_values(gdf, PATH_ELEV, 'elevation_m')

# --- 4) Add Mean Annual Temperature (WorldClim BIO1) ---
gdf = extract_raster_values(gdf, PATH_TEMP, 'temp_annual_c')

# --- 5) Add Annual Precipitation (WorldClim BIO12) ---
gdf = extract_raster_values(gdf, PATH_PREC, 'precip_annual_mm')

Extracting precip_annual_mm: 100%|██████████| 81973/81973 [00:19<00:00, 4266.98it/s]


In [71]:
# --- 6) Inspect correlations / missingness ---
print(gdf[['elevation_m','temp_annual_c','precip_annual_mm']].describe())
print("Missing elevation:", gdf['elevation_m'].isna().mean())

gdf = gdf.replace([-32768, -3.4028235e+38, -3.4e+37], np.nan)
gdf = gdf.dropna(subset=['elevation_m', 'temp_annual_c', 'precip_annual_mm'])

print(gdf['elevation_m'].describe())
print(gdf['temp_annual_c'].describe())
print(gdf['precip_annual_mm'].describe())

gdf.drop(['geometry'], axis=1, inplace=True)

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1025: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1025: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)


        elevation_m  temp_annual_c  precip_annual_mm
count  81973.000000   8.197300e+04      8.197300e+04
mean    -846.529138           -inf              -inf
std     5983.510693            inf               inf
min   -32768.000000  -3.400000e+38     -3.400000e+38
25%       10.000000   1.556250e+01      7.550000e+02
50%       58.000000   1.682917e+01      9.550000e+02
75%      164.000000   1.793750e+01      1.047000e+03
max     6700.000000   2.383750e+01      3.192000e+03
Missing elevation: 0.0
count    79215.000000
mean       264.869166
std        579.479893
min        -43.000000
25%         13.000000
50%         65.000000
75%        179.000000
max       6700.000000
Name: elevation_m, dtype: float64
count    79215.000000
mean        16.394686
std          3.501090
min        -15.770833
25%         15.845834
50%         17.112499
75%         18.008333
max         23.837500
Name: temp_annual_c, dtype: float64
count    79215.000000
mean       920.232361
std        321.536957
min         

# Exportación

In [72]:
gdf.to_csv(PATH_OUTPUT, index=False)
print(gdf.info())

for columna in df.columns:
    conteo = df[columna].value_counts()
    print(f"\nConteo de valores en la columna '{columna}':")
    print(conteo)

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 79215 entries, 2 to 83898
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order                          79215 non-null  object 
 1   family                         79215 non-null  object 
 2   genus                          79215 non-null  object 
 3   species                        79215 non-null  object 
 4   individualCount                79215 non-null  int64  
 5   decimalLatitude                79215 non-null  float64
 6   decimalLongitude               79215 non-null  float64
 7   coordinateUncertaintyInMeters  79215 non-null  float64
 8   day                            79215 non-null  int64  
 9   month                          79215 non-null  int64  
 10  timeRange                      79215 non-null  int64  
 11  hour                           79215 non-null  float64
 12  hour_cycle                     79215 non-nu